# Lab 07 · Security and privacy

Explore suspicious sign-ins, alert decisions, and ways to limit the information shared about households.

Fill each `____` using the comments, then run the cells in order.


## Feature Preparation


In [ ]:
# Import these packages and helpers using the names in the comments.
import numpy as ____                       # np
import pandas as ____                      # pd
from sklearn.model_selection import ____  # train_test_split
from sklearn.ensemble import ____  # RandomForestClassifier
from sklearn.impute import ____  # SimpleImputer
from sklearn.pipeline import ____  # Pipeline
from sklearn.metrics import ____  # Both metric names used below.
rng = np.random.default_rng(____)          # Fixed seed: 73.


In [ ]:
# Make 180 invented sign-ins: retries, unfamiliar device, and unusual hour.
retries = rng.integers(0, 7, size=____)
new_device = rng.integers(0, 2, size=____)
unusual_hour = rng.integers(0, 2, size=____)
review = (retries + 2 * new_device + unusual_hour >= ____).astype(int)  # Review when the score reaches 7.
records = pd.DataFrame({"retries": retries, "new_device": new_device, "unusual_hour": unusual_hour, "review": ____})  # Add the review labels.
records["account_alias"] = ["demo-" + str(i) for i in range(____)]
records.loc[7, "retries"] = ____       # One missing measurement.
print(records.____(3))                     # Replace ____ with head.


In [ ]:
# Remove the identifier; this reduces disclosure but does not prove anonymity.
safe_records = records.drop(columns=[____])
X = safe_records.drop(columns=[____])  # Keep the target out of features.
y = safe_records[____]
print(X.isna().____())                    # Missing values per feature.
print(y.value_counts(normalize=____))    # Class proportions.


In [ ]:
# Keep 20% for the final test, then 25% of the remainder for validation.
X_build, X_test, y_build, y_test = train_test_split(X, y, test_size=____, random_state=73, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_build, y_build, test_size=____, random_state=73, stratify=y_build)
print(len(X_train), len(____), len(X_test))


## Random Forests and Decision Thresholds


In [ ]:
# Learn missing-value replacements only from training rows.
detector = ____([                  # Combine imputation and forest in a Pipeline.
    ("fill", SimpleImputer(strategy=____)),  # Median replacement.
    ("forest", RandomForestClassifier(n_estimators=____, max_depth=2, random_state=73, class_weight="balanced"))])  # 24 trees, depth at most 2.
detector.____(X_train, y_train)
val_probability = detector.predict_proba(____)[:, 1]


In [ ]:
# Compare alert cutoffs on validation rows; a miss costs twice a false alert.
cutoffs = [____]               # Try 0.3, 0.5, and 0.7.
costs = ____                              # Empty list.
for cutoff in ____:
    alert = val_probability >= ____
    false_alerts = np.sum((y_val == 0) & (alert == ____))
    missed = np.sum((y_val == 1) & (alert == ____))
    costs.append(false_alerts + ____ * missed)
    print(cutoff, false_alerts, ____)


In [ ]:
# Freeze the lowest-cost cutoff before looking at the final test results.
chosen_cutoff = cutoffs[np.____(costs)]
test_probability = detector.predict_proba(____)[:, 1]
test_alert = (test_probability >= ____).astype(int)
print(confusion_matrix(y_test, test_alert, labels=[____]))  # Normal class first, review class second.
print(classification_report(y_test, test_alert, zero_division=____))  # Use 0 if a ratio is undefined.


## Feature Importance and Decision Trees


In [ ]:
# Rank the forest's feature importances; importance does not establish cause.
forest = detector.named_steps[____]
importance = pd.Series(forest.feature_importances_, index=____)
print(importance.sort_values(ascending=____))


In [ ]:
# Inspect only the first split in one tree, not a full forest explanation.
tree = forest.estimators_[____]
root_feature = tree.tree_.feature[____]
root_threshold = tree.tree_.threshold[____]
print(X.columns[____], root_threshold)
filled_example = detector.named_steps["fill"].transform(X_test.iloc[[____]])
go_left = filled_example[0, root_feature] <= ____
print("First row takes the left branch:", ____)


In [ ]:
# Compare two invented attempts; keep the training feature order.
attempts = pd.DataFrame([[0, 0, 0], [6, 1, 1]], columns=____)
review_chance = detector.predict_proba(____)[:, 1]
print(review_chance >= ____)
# These artificial labels cannot validate a real security detector.


## Anonymization and Differential Privacy


In [ ]:
# Each household may contribute at most one yes to a shared-device count.
reports = np.array([0, 3, 1, 0, 2, 1, 4, 0], dtype=____)  # Integer counts.
bounded = np.clip(reports, ____, ____)        # Cap each contribution at one.
true_count = bounded.____()
sensitivity = ____                       # Adding/removing one household changes count by at most 1.
print(true_count, ____)


In [ ]:
# Illustrate Laplace count noise: scale = sensitivity / epsilon.
# This fixed-seed exercise is reproducible, NOT a private data release.
epsilon = ____                           # Demonstration budget: 0.8.
noise_rng = np.random.default_rng(____)   # Fixed demonstration seed: 91.
noisy_count = true_count + noise_rng.laplace(0, sensitivity / ____)
print(____)
# Formal privacy needs a defined neighbouring-data rule and fresh suitable randomness.


In [ ]:
# Compare count error over many draws; the same base draws make the comparison clear.
unit_noise = noise_rng.laplace(0, 1, size=____)  # 2,000 draws.
budgets = [____]               # Compare 0.4, 0.8, and 1.6.
for budget in ____:
    simulated_counts = true_count + unit_noise * sensitivity / ____
    mean_error = np.mean(np.abs(simulated_counts - ____))
    print(budget, ____)


In [ ]:
# Repeated releases of the SAME people's data accumulate privacy expenditure.
# Under basic sequential composition, add the three release budgets.
release_budgets = np.array([____])  # Three budgets: 0.2, 0.3, 0.4.
combined_budget = release_budgets.____()
remaining_budget = ____ - combined_budget  # Planned overall limit: 1.0.
print(combined_budget, ____)
# Publishing this demonstration's exact count would defeat its illustrative protection.
